# P31 — Agentes generativos: simulacros interactivos de comportamiento humano

## 1. Título y paper

**Paper:** *Generative Agents: Interactive Simulacra of Human Behavior*  
**Autoría:** Joon Sung Park, Joseph C. O'Brien, Carrie J. Cai, Meredith Ringel Morris, Percy Liang, Michael S. Bernstein  
**Año y venue:** 2023 · arXiv:2304.03442 · UIST 2023  
**Nivel:** L3 · **Motor:** `generative_agents`  
**Ficha completa:** [`P31_generative_agents`](../../papers/foundational/P31_generative_agents/README.md)

**Hito:** Resuelve la memoria de un agente que vive mucho tiempo: qué recordar, cuándo y por qué, cuando el contexto no da para todo.

- [arXiv:2304.03442](https://arxiv.org/abs/2304.03442)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un agente con muchas horas de historia no cabe en su ventana de contexto, y un registro cronológico recupera lo reciente y trivial en vez de lo pertinente.
2. Ejecutar una implementación mínima de la propuesta: Un flujo de memoria con recuperación puntuada por relevancia, recencia e importancia, más un proceso de reflexión que sintetiza recuerdos en conclusiones de nivel superior.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P11
- P13
- P30


## 4. Intuición

Una persona no recuerda su día en orden cronológico: recuerda lo que viene a cuento. Si un agente guarda todo y recupera lo último, recordará que compró café en vez de que mañana hay una fiesta.


## 5. Concepto mínimo

```text
puntuación(recuerdo) = relevancia + recencia + importancia

    relevancia  = similitud con la consulta actual
    recencia    = decaimiento exponencial desde la última vez que se accedió
    importancia = cuán significativo es el recuerdo en sí (lo puntúa el modelo)
```

Y encima, **reflexión**: sintetizar periódicamente los recuerdos en conclusiones de nivel superior («Klaus está muy metido en su investigación»), que a su vez se guardan como recuerdos.


## 6. Código explicado

El motor puntúa cinco recuerdos ante una consulta, con las tres señales y solo con recencia.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('generative_agents', seed=7)['result']
print('consulta:', r['consulta'], '\n')
for m in r['ranking_completo']:
    print(f"{m['puntuacion']:.3f} = rel {m['relevancia']:.2f} + rec {m['recencia']:.3f} "
          f"+ imp {m['importancia']:.2f}  ← {m['texto']}")

## 7. Predicción antes de ejecutar

1. ¿Qué recuerdo debería recuperarse ante una consulta sobre la fiesta y Klaus?
2. ¿Cuál saldría si ordenáramos solo por lo más reciente?
3. ¿Qué señal evita que un recuerdo trivial y reciente gane?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
r = run_paper_lab('generative_agents', seed=7)['result']
print('con las tres señales :', r['top_con_las_tres_senales'])
print('solo por recencia    :', r['top_solo_por_recencia'])

## 9. Salida interpretable

Con las tres señales sube lo relevante e importante. Solo por recencia sube lo trivial. Una memoria útil **no es un registro cronológico**: es un sistema de recuperación con criterio, y ese criterio hay que diseñarlo.


## 10. Comentario pedagógico

Este paper se suele contar como «una simulación tipo Los Sims con LLM». Lo que importa técnicamente es otra cosa: es de los primeros que trata la **memoria de un agente de larga duración** como un problema de recuperación con puntuación, no como un log que se concatena.


## 11. Error o anti-patrón deliberado

Anti-patrón: meter toda la historia en el contexto porque «el contexto ya es grande».


In [ ]:
for horas in (1, 8, 24, 168):
    eventos = horas * 30
    tokens = eventos * 25
    print(f'{horas:>3} h de vida → ~{eventos:>5} eventos → ~{tokens:>7} tokens '
          f"({'cabe' if tokens < 100000 else 'imposible: hay que recuperar, no concatenar'})")

## 12. Corrección

La corrección es separar almacenamiento de recuperación, y puntuar:


In [ ]:
arquitectura = {
    'flujo_de_memoria': 'todo se guarda, con marca de tiempo e importancia',
    'recuperacion': 'se puntúa por relevancia + recencia + importancia y se toma el top-k',
    'reflexion': 'periódicamente se sintetizan recuerdos en conclusiones de alto nivel',
    'lo_que_entra_al_contexto': 'solo el top-k recuperado, no el flujo completo',
}
show(arquitectura)

## 13. Desafío guiado

Cambia el factor de decaimiento y observa cuánto pesa la recencia frente a la importancia.


In [ ]:
for decaimiento in (0.90, 0.99, 0.999):
    print(f'decaimiento {decaimiento}:')
    for antiguedad in (1, 10, 60):
        print(f'   hace {antiguedad:>2} pasos → recencia {decaimiento ** antiguedad:.4f}')

## 14. Desafío autónomo

Implementa un flujo de memoria con recuperación puntuada para un asistente que registre tu propia actividad durante una semana. Compara la utilidad de lo recuperado con las tres señales frente a solo similitud, sobre 20 consultas reales tuyas.


## 15. Evidencia de aprendizaje

Guarda el ranking con las tres señales, el que sale solo por recencia, y tu explicación de por qué concatenar el historial no escala.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P31_generative_agents/README.md) · evaluación formal: [`assessments/papers/P31_generative_agents.md`](../../assessments/papers/P31_generative_agents.md)


## 16. Cierre

El agente ya recuerda lo pertinente. Falta que lo aprendido se convierta en **capacidad reutilizable**, no solo en texto que recordar.


## 17. Conexión con el siguiente hito

- P16

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
